In [2]:
import numpy as np
import torch
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt

%matplotlib inline
torch.set_printoptions(sci_mode=False,precision=4)
torch.manual_seed(123)
torch.random.manual_seed(123)
generator = torch.Generator().manual_seed(123)

In [3]:
context_length = 96
batch_size = 32
embd_dim = 32
iteration_count = 100000

In [4]:
with open('../data/amir_khan.txt', 'r', encoding = 'utf-8') as file:
    records = file.read()

In [5]:
data = ''.join(records)
data = '_' * context_length + data
unique_text = sorted(list(set(data)))
vocab_size = len(unique_text)

In [6]:
stoi = {text: index for index, text in enumerate(unique_text)}
itos = {index: text for index, text in enumerate(unique_text)}

encoder = lambda char_array: [stoi[char] for char in ''.join(char_array)]
decoder = lambda num_array: [itos[num] for num in num_array]

In [7]:
x_numpy = []
y_numpy = []
temp_data = data

for t in range(len(temp_data) - context_length - 1):
    x_numpy.append(encoder(temp_data[t : t + context_length]))
    y_numpy.append(stoi[temp_data[t + context_length]])

x = torch.tensor(x_numpy, dtype=torch.long)
y = torch.tensor(y_numpy, dtype=torch.long)



N = x.shape[0]
train_ratio = 0.8
val_ratio = 0.1

train_end = int(train_ratio * N)
val_end = int((train_ratio + val_ratio) * N)

x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:], y[train_end:]

In [8]:
criterion = nn.CrossEntropyLoss()
embedding = nn.Embedding(vocab_size, embd_dim)
logit_layer = nn.Linear(context_length * embd_dim, vocab_size)

head_size = 12
embedding = nn.Embedding(vocab_size, embd_dim) # Assuming vocab_size is defined
key = nn.Linear(embd_dim, head_size, bias=False)
query = nn.Linear(embd_dim, head_size, bias=False)
value = nn.Linear(embd_dim, head_size, bias=False)
logit_layer = nn.Linear(head_size * context_length, vocab_size) # Adjusted input size

In [10]:
def evaluate(x_data, y_data):
    # Set all layers to evaluation mode
    embedding.eval()
    key.eval()
    query.eval()
    value.eval()
    logit_layer.eval()

    with torch.no_grad():
        # 1. Embedding
        x = embedding(x_data) # (B, T, Embd_Size)

        # 2. Self-Attention (Must match training logic exactly)
        k = key(x)   # (B, T, head_size)
        q = query(x) # (B, T, head_size)

        # Calculate attention scores
        # Note: 'tril' should be accessible globally or passed in
        T = x.size(1)
        wei = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5)
        # Use a local mask in case T varies
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        wei = wei.masked_fill(mask == 0, float('-inf'))
        wei = torch.nn.functional.softmax(wei, dim=-1)

        # Apply weights to values
        v = value(x)
        out = wei @ v # (B, T, head_size)

        # 3. Flatten and get Logits
        # Ensure this view matches the shape used in training
        out_flattened = out.reshape(out.size(0), -1)
        logits = logit_layer(out_flattened)

        # 4. Metrics
        val_loss = criterion(logits, y_data)
        predictions = torch.argmax(logits, dim=1) # softmax not needed for argmax
        accuracy = (predictions == y_data).float().mean()

    # Set all layers back to training mode
    embedding.train()
    key.train()
    query.train()
    value.train()
    logit_layer.train()

    return val_loss.item(), accuracy.item()

In [14]:
# --- Optimizer ---
# Note: embedding and logit_layer should be in .train() mode for training
optimizer = optim.Adam(
    list(embedding.parameters()) +
    list(logit_layer.parameters()) +
    list(key.parameters()) +
    list(query.parameters()) +
    list(value.parameters()),
    lr=0.005
)

tril = torch.tril(torch.ones(context_length, context_length))

for epoch in range(1):
    # 1. Data Fetching
    idx = torch.randint(0, len(x_train), (batch_size,), generator=generator)
    rand_x = x_train[idx] # Shape: (batch, context_length)
    rand_y = y_train[idx]

    optimizer.zero_grad(set_to_none=True)

    # 2. Embedding
    x = embedding(rand_x) # Shape: (batch, context_length, embd_size)

    # 3. Self-Attention Mechanism
    k = key(x)   # (B, T, head_size)
    q = query(x) # (B, T, head_size)

    # Calculate attention scores (affinities)
    wei = q @ k.transpose(-2, -1) * (head_size**-0.5) # Scaled dot-product
    wei = wei.masked_fill(tril == 0, float('-inf'))
    wei = torch.nn.functional.softmax(wei, dim=-1)

    # Apply weights to values
    v = value(x) # (B, T, head_size)
    out = wei @ v # (B, T, head_size)

    print(out.shape)

    # 4. Preparation for Logit Layer
    # Flatten the sequence: (B, T, H) -> (B, T*H)
    out_flattened = out.view(batch_size, -1)

    print(out_flattened.shape)

    logits = logit_layer(out_flattened)
    print('logits.shape', logits.shape)
    print('rand_y' , rand_y.shape)
    loss = criterion(logits, rand_y)

    # 5. Backprop
    if epoch % 10000 == 0:
        print(loss.item(), evaluate(x_val, y_val)[0])

    loss.backward()
    optimizer.step()

torch.Size([32, 96, 12])
torch.Size([32, 1152])
logits.shape torch.Size([32, 66])
rand_y torch.Size([32])
3.8185131549835205 3.5930168628692627
